# Notebook 09 — LightGBM Pre-Ranking

**Objective:** Train a supervised *candidate re-ranker* on Notebook 08's hybrid candidates, using the **August 1** historical snapshot as features and August 1–14 future views as implicit positive labels. Evaluate on held-out **validation users** and then on the untouched **September test** snapshot.

**Important evaluation distinction:** The validation cohort's future views are used for training (except held-out users). Therefore the full validation set is **not** a valid independent metric for this model. The held-out user subset is the validation metric; September is the final test. Model selection is done on the held-out validation users only. This is a *user-disjoint holdout within one time window*, not a second chronological window.

No impression logs exist: non-clicked/unobserved candidates are **unlabeled**, treated as sampled implicit negatives for this experiment, not verified negative impressions. The fixed Top-100 retrieval set puts an upper bound on ranker recall. Training uses candidate features available before the corresponding cutoff; never future events as features.

Install once if needed: `pip install lightgbm scikit-learn pandas numpy`.

## 1. Load reproducible retrieval exports and histories

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd

cwd=Path.cwd().resolve()
if (cwd/'data'/'processed').is_dir(): ROOT=cwd
elif (cwd.parent/'data'/'processed').is_dir(): ROOT=cwd.parent
else: raise FileNotFoundError('Cannot locate project data/processed')
PROC=ROOT/'data'/'processed'; REPORTS=ROOT/'reports'; REPORTS.mkdir(exist_ok=True)
import lightgbm as lgb
from sklearn.model_selection import train_test_split

SEED=42; K_VALUES=[10,20,50,100]; MAX_K=100
CUT={'val':pd.Timestamp('2015-08-01',tz='UTC'), 'test':pd.Timestamp('2015-09-01',tz='UTC')}
raw={}
for split,stem,hfile in [('val','validation','train_history.csv'),('test','test','test_history.csv')]:
    cand=pd.read_csv(PROC/f'notebook08_{split}_hybrid_candidates.csv',dtype={'visitorid':'int64','rank':'int16','itemid':'int64'})
    hist=pd.read_csv(PROC/hfile,usecols=['timestamp','visitorid','event','itemid'])
    labels=pd.read_csv(PROC/f'{stem}_view_labels_warm_users.csv',usecols=['visitorid','itemid'])
    assert hist.timestamp.max()<int(CUT[split].timestamp()*1000)
    assert not cand[['visitorid','itemid']].duplicated().any()
    assert not cand[['visitorid','rank']].duplicated().any()
    assert cand.groupby('visitorid').size().eq(MAX_K).all()
    assert set(cand.visitorid.unique())==set(labels.visitorid.unique())
    assert set(['from_itemcf','from_faiss_fallback','from_popularity']).issubset(cand.columns)
    raw[split]=(cand,hist,labels)
    print(split,'candidate rows',len(cand),'history',len(hist),'positive labels',len(labels))

val candidate rows 1332900 history 1899029 positive labels 28634
test candidate rows 1513700 history 2452391 positive labels 31876


## 2. Build strictly historical ranking features

All counts are computed separately from each split's **pre-cutoff** history. We do **not** join full-dataset activity or future labels as features. `rank` is the Hybrid RRF rank, not a target; source flags are from the Notebook 08 export. Recency is in days relative to the cutoff. Feature extraction uses only the current user's pre-cutoff behavior and global pre-cutoff item statistics.

In [2]:
FEATURES=['retrieval_rank','rrf_inverse_rank','from_itemcf','from_faiss_fallback','from_popularity',
          'source_count','log_item_views','log_item_unique_viewers','log_user_views',
          'log_user_unique_items','user_last_event_days','item_last_view_days',
          'item_view_share','log_item_carts','log_item_purchases']

def make_features(candidates, history, cutoff):
    views=history.loc[history.event.eq('view'),['visitorid','itemid','timestamp']]
    stats=views.groupby('itemid',sort=False).agg(item_views=('visitorid','size'),
          item_unique_viewers=('visitorid','nunique'),item_last_view=('timestamp','max'))
    users=views.groupby('visitorid',sort=False).agg(user_views=('itemid','size'),
          user_unique_items=('itemid','nunique'),user_last_event=('timestamp','max'))
    other=history.loc[history.event.isin(['addtocart','transaction']),['itemid','event']]
    if len(other):
        event_counts=other.groupby(['itemid','event']).size().unstack(fill_value=0)
        stats['item_carts']=event_counts['addtocart'] if 'addtocart' in event_counts else 0
        stats['item_purchases']=event_counts['transaction'] if 'transaction' in event_counts else 0
    else: stats['item_carts']=0;stats['item_purchases']=0
    stats[['item_carts','item_purchases']]=stats[['item_carts','item_purchases']].fillna(0)
    d=candidates.merge(stats,left_on='itemid',right_index=True,how='left',validate='many_to_one')
    d=d.merge(users,left_on='visitorid',right_index=True,how='left',validate='many_to_one')
    ms=int(cutoff.timestamp()*1000)
    d['retrieval_rank']=d['rank'].astype('float32')
    d['rrf_inverse_rank']=(1.0/(60+d['rank'])).astype('float32')
    d['source_count']=d[['from_itemcf','from_faiss_fallback','from_popularity']].sum(axis=1)
    for src,dst in [('item_views','log_item_views'),('item_unique_viewers','log_item_unique_viewers'),
                    ('user_views','log_user_views'),('user_unique_items','log_user_unique_items'),
                    ('item_carts','log_item_carts'),('item_purchases','log_item_purchases')]:
        d[dst]=np.log1p(d[src].fillna(0)).astype('float32')
    d['user_last_event_days']=((ms-d.user_last_event)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_last_view_days']=((ms-d.item_last_view)/86400000).clip(lower=0).fillna(999).astype('float32')
    d['item_view_share']=(d.item_views.fillna(0)/max(1,len(views))).astype('float32')
    for name in FEATURES: d[name]=d[name].fillna(0).astype('float32')
    return d[['visitorid','itemid','rank']+FEATURES].copy()

features={}
for split in ['val','test']:
    cand,hist,labels=raw[split]
    t=time.perf_counter()
    features[split]=make_features(cand,hist,CUT[split])
    assert features[split][FEATURES].notna().all().all()
    print(split,'feature matrix',features[split].shape,'seconds',round(time.perf_counter()-t,2))

val feature matrix (1332900, 18) seconds 2.03
test feature matrix (1513700, 18) seconds 2.66


## 3. Construct implicit-feedback labels and user-disjoint validation split

We train with **all candidate positives** and at most 20 randomly sampled unobserved candidates per training user. Users with zero candidate positives still contribute unlabeled candidates. No candidate is assumed to be an actual shown impression. Hold out 20% of validation users before fitting; use the held-out users to select iterations via early stopping.

In [3]:
def attach_labels(df, labels):
    positives=labels[['visitorid','itemid']].drop_duplicates().assign(label=np.int8(1))
    out=df.merge(positives,on=['visitorid','itemid'],how='left',validate='one_to_one')
    out['label']=out['label'].fillna(0).astype('int8')
    return out

val=attach_labels(features['val'],raw['val'][2]);test=attach_labels(features['test'],raw['test'][2])
all_users=np.sort(val.visitorid.unique())
train_users,holdout_users=train_test_split(all_users,test_size=.20,random_state=SEED)
train_users=set(map(int,train_users));holdout_users=set(map(int,holdout_users))
assert not(train_users & holdout_users)

# Deterministic negative sampling by user; retain all positives and at most 20 implicit negatives.
def sample_train_rows(frame, user_ids, neg_per_user=20, seed=SEED):
    sub=frame.loc[frame.visitorid.isin(user_ids)].copy()
    positives=sub[sub.label.eq(1)]
    negatives=sub[sub.label.eq(0)]
    negatives=(negatives.assign(_random=np.random.default_rng(seed).random(len(negatives)))
              .sort_values(['visitorid','_random'],kind='mergesort')
              .groupby('visitorid',sort=False).head(neg_per_user).drop(columns='_random'))
    return pd.concat([positives,negatives],ignore_index=True).sort_values(['visitorid','rank'],kind='mergesort').reset_index(drop=True)

train_df=sample_train_rows(val,train_users)
valid_df=val.loc[val.visitorid.isin(holdout_users)].sort_values(['visitorid','rank']).reset_index(drop=True)
assert train_df.visitorid.nunique()==len(train_users)
print('Training candidates:',len(train_df),'positive:',int(train_df.label.sum()))
print('Held-out candidates:',len(valid_df),'positive:',int(valid_df.label.sum()))
print('Test candidates:',len(test),'positive:',int(test.label.sum()))

Training candidates: 214772 positive: 1512
Held-out candidates: 266600 positive: 376
Test candidates: 1513700 positive: 1829


## 4. Train LightGBM binary pre-ranker

Use binary cross-entropy on sampled implicit negatives as a pragmatic first pre-ranking model. Its output is a **relative ranking score**, not calibrated CTR. We do not evaluate model quality on the same users used for fitting.

In [4]:
params=dict(objective='binary',learning_rate=0.05,num_leaves=31,max_depth=-1,
            min_child_samples=50,colsample_bytree=0.9,subsample=0.9,subsample_freq=1,
            reg_lambda=2.0,random_state=SEED,n_jobs=-1,verbosity=-1)
model=lgb.LGBMClassifier(n_estimators=600,**params)
t0=time.perf_counter()
model.fit(train_df[FEATURES],train_df.label,
          eval_set=[(valid_df[FEATURES],valid_df.label)],eval_metric='binary_logloss',
          callbacks=[lgb.early_stopping(35,verbose=False)])
print('Fit seconds:',round(time.perf_counter()-t0,2),'best iteration:',model.best_iteration_)
assert model.best_iteration_ is not None
importance=(pd.DataFrame({'feature':FEATURES,'importance':model.feature_importances_})
            .sort_values('importance',ascending=False))
display(importance)

Fit seconds: 3.83 best iteration: 427


,feature,importance
10,user_last_event_days,2628
11,item_last_view_days,2188
6,log_item_views,1414
0,retrieval_rank,1353
7,log_item_unique_viewers,1342
8,log_user_views,1155
9,log_user_unique_items,916
13,log_item_carts,595
1,rrf_inverse_rank,507
14,log_item_purchases,315


## 5. Evaluate ranker vs the original Hybrid ordering

The retrieval pool is unchanged. **Recall@100 must be identical** between Hybrid and LightGBM; improvement is possible only at smaller K. We report both candidate-conditioned recall and end-to-end recall using **all future-view labels for each evaluated user**, including positives outside the candidate pool. Avoid interpreting this as CTR.

In [5]:
def rank_candidates(frame, scores):
    d=frame[['visitorid','itemid','rank']].copy()
    d['score']=np.asarray(scores,dtype='float32')
    d=d.sort_values(['visitorid','score','rank','itemid'],ascending=[True,False,True,True],kind='mergesort')
    d['new_rank']=d.groupby('visitorid',sort=False).cumcount()+1
    return d

def evaluate_ranks(frame, labels, rank_column, k_values=K_VALUES):
    # Use complete label sets, not only positives present in candidate table.
    truth=labels.groupby('visitorid').itemid.agg(lambda x:set(map(int,x))).to_dict()
    f=frame.sort_values(['visitorid',rank_column],kind='mergesort')
    recs=f.groupby('visitorid',sort=False).itemid.agg(list).to_dict()
    assert set(recs)==set(truth)
    eligible_hits=sum(len(set(items)&truth[int(u)]) for u,items in recs.items())
    rows=[]
    for k in k_values:
        found=0;hit_users=0;macro=[]
        for u,items in recs.items():
            hits=len(set(items[:k])&truth[int(u)])
            found+=hits;hit_users+=int(hits>0);macro.append(hits/len(truth[int(u)]))
        rows.append(dict(K=k,Recall_micro=found/sum(map(len,truth.values())),
                         Recall_macro=float(np.mean(macro)),HitRate=hit_users/len(truth),
                         CandidateConditionedRecall=found/eligible_hits if eligible_hits else np.nan,
                         hits=found))
    return pd.DataFrame(rows)

valid_ranked=rank_candidates(valid_df,model.predict_proba(valid_df[FEATURES])[:,1])
base_valid=evaluate_ranks(valid_df,raw['val'][2].loc[raw['val'][2].visitorid.isin(holdout_users)],'rank')
new_valid=evaluate_ranks(valid_ranked,raw['val'][2].loc[raw['val'][2].visitorid.isin(holdout_users)],'new_rank')
comparison=base_valid.merge(new_valid,on='K',suffixes=('_hybrid','_lightgbm'))
assert np.isclose(comparison.iloc[-1].Recall_micro_hybrid,comparison.iloc[-1].Recall_micro_lightgbm)
display(comparison[['K','Recall_micro_hybrid','Recall_micro_lightgbm','HitRate_hybrid','HitRate_lightgbm','CandidateConditionedRecall_lightgbm']])

,K,Recall_micro_hybrid,Recall_micro_lightgbm,HitRate_hybrid,HitRate_lightgbm,CandidateConditionedRecall_lightgbm
0,10,0.025740,0.032270,0.045011,0.054014,0.446809
1,20,0.038033,0.046101,0.063016,0.071643,0.638298
2,50,0.054552,0.061468,0.081770,0.089272,0.851064
3,100,0.072224,0.072224,0.100525,0.100525,1.000000


## 6. Refit using all August validation users; evaluate once on September test

After selecting the number of trees on the held-out August user subset, refit using all August users. The September test labels are used **only** for final evaluation. This is valid as a time-forward experiment: August 1–14 outcomes are available before the September 1 cutoff. No hyperparameters should be changed based on the September results.

In [6]:
all_aug=sample_train_rows(val,set(map(int,all_users)),neg_per_user=20)
final_model=lgb.LGBMClassifier(n_estimators=max(1,int(model.best_iteration_)),**params)
t0=time.perf_counter()
final_model.fit(all_aug[FEATURES],all_aug.label)
print('Final fit seconds:',round(time.perf_counter()-t0,2))

t0=time.perf_counter()
test_scores=final_model.predict_proba(test[FEATURES])[:,1]
score_seconds=time.perf_counter()-t0
test_ranked=rank_candidates(test,test_scores)
base_test=evaluate_ranks(test,raw['test'][2],'rank')
new_test=evaluate_ranks(test_ranked,raw['test'][2],'new_rank')
assert np.isclose(base_test.iloc[-1].Recall_micro,new_test.iloc[-1].Recall_micro)
test_comparison=base_test.merge(new_test,on='K',suffixes=('_hybrid','_lightgbm'))
print('Test scoring seconds:',round(score_seconds,3))
display(test_comparison[['K','Recall_micro_hybrid','Recall_micro_lightgbm','HitRate_hybrid','HitRate_lightgbm','CandidateConditionedRecall_lightgbm']])

Final fit seconds: 2.69
Test scoring seconds: 4.854


,K,Recall_micro_hybrid,Recall_micro_lightgbm,HitRate_hybrid,HitRate_lightgbm,CandidateConditionedRecall_lightgbm
0,10,0.020768,0.026383,0.037590,0.047632,0.459814
1,20,0.029427,0.036767,0.051133,0.062232,0.640787
2,50,0.045018,0.049881,0.073198,0.079804,0.869328
3,100,0.057379,0.057379,0.088855,0.088855,1.000000


## 7. Save model, scored candidates, evaluation report, and checks

**Important:** Training and evaluation labels exist only in offline artifacts. Do not export `label` as a serving feature. The model is trained using the August cohort and scores September candidates using the September pre-cutoff feature snapshot. Later notebooks can use the scored candidate CSV for Fine Ranking experiments, but avoid evaluating a new model on the same data it trained on.

In [7]:
import pickle
model_path=REPORTS/'notebook09_lightgbm_preranker.pkl'
with open(model_path,'wb') as f:pickle.dump({'model':final_model,'features':FEATURES,'cutoff_training':'2015-08-01','label_window':'2015-08-01/2015-08-15'},f)

valid_ranked.to_csv(PROC/'notebook09_val_holdout_scored_candidates.csv',index=False)
test_ranked.to_csv(PROC/'notebook09_test_scored_candidates.csv',index=False)
comparison.to_csv(REPORTS/'notebook09_val_holdout_metrics.csv',index=False)
test_comparison.to_csv(REPORTS/'notebook09_test_metrics.csv',index=False)
importance.to_csv(REPORTS/'notebook09_feature_importance.csv',index=False)

summary={'method':'LightGBM sampled-implicit-negative binary candidate preranker',
         'protocol':'Aug warm-user cohort, user-disjoint 80/20 holdout; refit on full Aug; evaluate Sep once',
         'caveats':['No impression logs; unobserved candidates are not confirmed negatives',
                    'Candidate pool fixed to Notebook 08 Top-100',
                    'Full August validation metrics are in-sample after final refit and are not reported'],
         'features':FEATURES,'best_iteration':int(model.best_iteration_),
         'train_rows':int(len(train_df)),'train_positives':int(train_df.label.sum()),
         'validation_holdout_users':int(len(holdout_users)),
         'validation_holdout':comparison.to_dict(orient='records'),
         'test':test_comparison.to_dict(orient='records'),
         'test_scoring_seconds':float(score_seconds)}
path=REPORTS/'notebook09_lightgbm_preranking.json'
path.write_text(json.dumps(summary,indent=2),encoding='utf-8')
assert not any('label' in x for x in FEATURES)
assert all(test_ranked.groupby('visitorid').size().eq(100))
assert test_ranked.groupby('visitorid').new_rank.max().eq(100).all()
assert np.isclose(test_comparison.loc[test_comparison.K.eq(100),'Recall_micro_hybrid'].iloc[0],
                  test_comparison.loc[test_comparison.K.eq(100),'Recall_micro_lightgbm'].iloc[0])
print('Saved:',path)
print('Saved:',model_path)
print('PASS: temporal feature snapshots, user-disjoint holdout, test-only final evaluation, and fixed candidate-pool checks')

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook09_lightgbm_preranking.json
Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook09_lightgbm_preranker.pkl
PASS: temporal feature snapshots, user-disjoint holdout, test-only final evaluation, and fixed candidate-pool checks


## 8. Interpretation and next notebook

- Compare **Recall@10, @20, @50** on held-out August users and September test users; **Recall@100 cannot improve** when reordering the same Top-100 pool.
- Low positive prevalence and missing impressions make raw probability calibration unreliable.
- Feature importance indicates usage, not causal impact.
- Next: **Notebook 10 — DeepFM Fine Ranking**, using a separate time-aware training protocol and careful comparison against the LightGBM ranker.